In [1]:
import numpy as np


def layer_norm(input_embeddings, layer_norm_scale, layer_norm_shift, epsilon=1e-5):

    feature_mean = np.mean(input_embeddings, axis=-1, keepdims=True)
    feature_variance = np.var(input_embeddings, axis=-1, keepdims=True)

    normalized_embeddings = (input_embeddings - feature_mean) / np.sqrt(feature_variance + epsilon)

    return layer_norm_scale * normalized_embeddings + layer_norm_shift

def multi_head_attention(query_states, key_value_states,
                         query_projection_weights, key_projection_weights,
                         value_projection_weights, output_projection_weights,
                         number_of_heads, use_causal_mask=False):

    query_sequence_length, model_dimension = query_states.shape
    key_sequence_length = key_value_states.shape[0]

    head_dimension = model_dimension // number_of_heads

    assert model_dimension % number_of_heads == 0

    #  1: Linear projections
    query = query_states @ query_projection_weights
    key = key_value_states @ key_projection_weights
    value = key_value_states @ value_projection_weights

    query = query.reshape(query_sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    key = key.reshape(key_sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    value = value.reshape(key_sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)

    attention_scores = query @ key.transpose(0, 2, 1) / np.sqrt(head_dimension)

    if use_causal_mask:
        causal_mask = np.triu(np.ones((query_sequence_length, key_sequence_length)), k=1)
        attention_scores = np.where(causal_mask == 1, -np.inf, attention_scores)

    attention_scores = attention_scores - np.max(attention_scores, axis=-1, keepdims=True)

    exponentiated_scores = np.exp(attention_scores)

    attention_weights = exponentiated_scores / np.sum(
        exponentiated_scores, axis=-1, keepdims=True
    )

    per_head_outputs = attention_weights @ value

    concatenated_heads = per_head_outputs.transpose(1, 0, 2).reshape(
        query_sequence_length, model_dimension
    )

    attention_output = concatenated_heads @ output_projection_weights

    return attention_output, attention_weights

def feed_forward(input_embeddings, feed_forward_weights_1, feed_forward_bias_1,
                 feed_forward_weights_2, feed_forward_bias_2):

    hidden_activations = np.maximum(
        0, input_embeddings @ feed_forward_weights_1 + feed_forward_bias_1
    )  # ReLU

    feed_forward_output = hidden_activations @ feed_forward_weights_2 + feed_forward_bias_2

    return feed_forward_output

In [2]:
def transformer_decoder_block(decoder_input_embeddings,
                              encoder_output_embeddings, model_parameters):
    """
    Return decoder output after:
    1. Masked self-attention
    2. Residual + normalization
    3. Cross-attention
    4. Residual + normalization
    5. Feed-forward network
    6. Residual + normalization
    """

    # 1. Masked Multi-Head Self-Attention
    self_attention_output, self_attention_weights = multi_head_attention(
        decoder_input_embeddings,
        decoder_input_embeddings,
        model_parameters['query_projection_weights_self'],
        model_parameters['key_projection_weights_self'],
        model_parameters['value_projection_weights_self'],
        model_parameters['output_projection_weights_self'],
        model_parameters['number_of_heads'],
        use_causal_mask=True
    )

    # 2. Residual + Layer Normalization
    intermediate_decoder_embeddings = layer_norm(
        decoder_input_embeddings + self_attention_output,
        model_parameters['layer_norm_scale_1'],
        model_parameters['layer_norm_shift_1']
    )

    # 3. Cross-Attention
    cross_attention_output, cross_attention_weights = multi_head_attention(
        intermediate_decoder_embeddings,
        encoder_output_embeddings,
        model_parameters['query_projection_weights_cross'],
        model_parameters['key_projection_weights_cross'],
        model_parameters['value_projection_weights_cross'],
        model_parameters['output_projection_weights_cross'],
        model_parameters['number_of_heads'],
        use_causal_mask=False
    )

    # 4. Residual + Layer Normalization
    intermediate_decoder_embeddings_2 = layer_norm(
        intermediate_decoder_embeddings + cross_attention_output,
        model_parameters['layer_norm_scale_2'],
        model_parameters['layer_norm_shift_2']
    )

    # 5. Feed Forward Network
    feed_forward_output = feed_forward(
        intermediate_decoder_embeddings_2,
        model_parameters['feed_forward_weights_1'],
        model_parameters['feed_forward_bias_1'],
        model_parameters['feed_forward_weights_2'],
        model_parameters['feed_forward_bias_2']
    )

    # 6. Residual + Layer Normalization
    decoder_output = layer_norm(
        intermediate_decoder_embeddings_2 + feed_forward_output,
        model_parameters['layer_norm_scale_3'],
        model_parameters['layer_norm_shift_3']
    )

    return decoder_output, self_attention_weights, cross_attention_weights

In [3]:
model_dimension = 8
number_of_heads = 2
feed_forward_dimension = 16

encoder_sequence_length = 5
decoder_sequence_length = 3

# Encoder and Decoder inputs
sample_encoder_output_embeddings = np.random.randn(encoder_sequence_length, model_dimension)
sample_decoder_input_embeddings = np.random.randn(decoder_sequence_length, model_dimension)

model_parameters = {

    # Self-attention weights
    'query_projection_weights_self': np.random.randn(model_dimension, model_dimension) * 0.1,
    'key_projection_weights_self': np.random.randn(model_dimension, model_dimension) * 0.1,
    'value_projection_weights_self': np.random.randn(model_dimension, model_dimension) * 0.1,
    'output_projection_weights_self': np.random.randn(model_dimension, model_dimension) * 0.1,

    # Cross-attention weights
    'query_projection_weights_cross': np.random.randn(model_dimension, model_dimension) * 0.1,
    'key_projection_weights_cross': np.random.randn(model_dimension, model_dimension) * 0.1,
    'value_projection_weights_cross': np.random.randn(model_dimension, model_dimension) * 0.1,
    'output_projection_weights_cross': np.random.randn(model_dimension, model_dimension) * 0.1,

    # Feed-forward weights
    'feed_forward_weights_1': np.random.randn(model_dimension, feed_forward_dimension) * 0.1,
    'feed_forward_bias_1': np.zeros(feed_forward_dimension),

    'feed_forward_weights_2': np.random.randn(feed_forward_dimension, model_dimension) * 0.1,
    'feed_forward_bias_2': np.zeros(model_dimension),

    # Layer normalization parameters
    'layer_norm_scale_1': np.ones(model_dimension),
    'layer_norm_shift_1': np.zeros(model_dimension),

    'layer_norm_scale_2': np.ones(model_dimension),
    'layer_norm_shift_2': np.zeros(model_dimension),

    'layer_norm_scale_3': np.ones(model_dimension),
    'layer_norm_shift_3': np.zeros(model_dimension),

    'number_of_heads': number_of_heads
}

(decoder_output, self_attention_weights,
 cross_attention_weights) = transformer_decoder_block(
    sample_decoder_input_embeddings,
    sample_encoder_output_embeddings,
    model_parameters
)

print("Encoder Input Shape:", sample_encoder_output_embeddings.shape)
print("Decoder Input Shape:", sample_decoder_input_embeddings.shape)

print("\nSelf-Attention Weights:", self_attention_weights.shape)
print("Cross-Attention Weights:", cross_attention_weights.shape)

print("\nFinal Decoder Output Shape:", decoder_output.shape)

# Verify attention normalization
print("\nSelf-Attention Row Sums:")
print(self_attention_weights.sum(axis=-1))

print("\nCross-Attention Row Sums:")
print(cross_attention_weights.sum(axis=-1))

Encoder Input Shape: (5, 8)
Decoder Input Shape: (3, 8)

Self-Attention Weights: (2, 3, 3)
Cross-Attention Weights: (2, 3, 5)

Final Decoder Output Shape: (3, 8)

Self-Attention Row Sums:
[[1. 1. 1.]
 [1. 1. 1.]]

Cross-Attention Row Sums:
[[1. 1. 1.]
 [1. 1. 1.]]
